In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import openmm
import openmmtorch
from openmm import unit, Platform
from openmm.app import *
from openmm.unit import *
from openff.toolkit.topology import Molecule
from openmmforcefields.generators import EspalomaTemplateGenerator
import resff
import espaloma as esp
from residual.models.model import load_model
from typing import Iterable, Optional, Tuple

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

LIGAND_PATH = "./data/tetrapeptide.sdf"
RESIDUAL_WEIGHTS = "./residual.ckpt"
MM_WEIGHTS = "./MM.pt"

# =========================
# RESIDUAL MODULE INTEGRATION
# =========================

class Wrapper(torch.nn.Module):

    def __init__(self, topology, embeddings, mol_atom_indices, model, isPeriodic):
        super(Wrapper, self).__init__()
        self.embeddings = embeddings
        self.mol_atom_indices = mol_atom_indices

        if isPeriodic:
            self.box_vectors = topology.getPeriodicBoxVectors()
            self.box_vectors_np = 10.0 * np.array([[self.box_vectors[0][0].value_in_unit(nanometer), self.box_vectors[0][1].value_in_unit(nanometer), self.box_vectors[0][2].value_in_unit(nanometer)],
                                    [self.box_vectors[1][0].value_in_unit(nanometer), self.box_vectors[1][1].value_in_unit(nanometer), self.box_vectors[1][2].value_in_unit(nanometer)],
                                    [self.box_vectors[2][0].value_in_unit(nanometer), self.box_vectors[2][1].value_in_unit(nanometer), self.box_vectors[2][2].value_in_unit(nanometer)]])
        else:
            self.box_vectors_np = None

        # OpenMM will compute the forces by backpropagating the energy,
        # so we can load the model with derivative=False
        self.model = load_model(model, derivative=False, max_num_neighbors=128,cutoff_upper=10.0, box_vecs=self.box_vectors_np).to(torch.device("cuda"))

    def forward(self, positions, boxvectors: Optional[torch.Tensor] = None):
        # OpenMM works with nanometer positions and kilojoule per mole energies
        # Depending on the model, you might need to convert the units
        positions = positions[:self.mol_atom_indices, :]
        positions = positions.to(torch.float32) * 10.0 # nm -> A
        positions = positions.to(torch.device("cuda"))
        if boxvectors is not None:
            boxvectors = boxvectors.to(torch.float32) * 10.0
            boxvectors = boxvectors.to(torch.device("cuda"))
        energy = self.model(z=self.embeddings, pos=positions, box=boxvectors)[0]
        return energy * 4.184 # kcal/mol -> kJ/mol


# =========================
# Build System
# =========================

print("Loading ligand...")
ligand = Molecule.from_file(LIGAND_PATH)
graph = esp.Graph(ligand)
embeddings = graph.nodes["n1"].data["h0"].to("cuda")

openmm_topology = ligand.to_topology().to_openmm()
openmm_positions = ligand.conformers[0].to_openmm()

# Base force field
forcefield = ForceField(
    "amber/protein.ff14SB.xml",
    "amber14/tip3pfb.xml",
)

# Register Espaloma
esp_generator = EspalomaTemplateGenerator(
    molecules=ligand,
    forcefield=MM_WEIGHTS,
)
forcefield.registerTemplateGenerator(esp_generator.generator)

modeller = Modeller(openmm_topology, openmm_positions)

# Solvate system
modeller.addSolvent(forcefield, padding=1.0 * nanometer)

system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=0.9 * nanometer,
    constraints=HBonds,
)

is_periodic = system.usesPeriodicBoundaryConditions()
print("Periodic:", is_periodic)


# =========================
# Add TorchForce
# =========================

n_ligand_atoms = len(list(openmm_topology.atoms()))

wrapper = Wrapper(
    modeller.topology,
    embeddings,
    n_ligand_atoms,
    RESIDUAL_WEIGHTS,
    is_periodic,
)

scripted_model = torch.jit.script(wrapper)
torch_force = openmmtorch.TorchForce(scripted_model)
torch_force.setUsesPeriodicBoundaryConditions(is_periodic)
system.addForce(torch_force)


# =========================
# Minimization
# =========================

def describe_state(state, name="State"):
    forces = state.getForces(asNumpy=True)
    max_force = np.max(np.linalg.norm(forces, axis=1))
    energy = state.getPotentialEnergy().value_in_unit(kilojoule_per_mole)

    print(f"{name}: Energy = {energy:.2f} kJ/mol | Max force = {max_force:.2f}")


integrator = openmm.LangevinIntegrator(
    500 * kelvin,
    1.0 / picoseconds,
    0.001 * picoseconds
)

platform = Platform.getPlatformByName("CUDA")

simulation = Simulation(
    modeller.topology,
    system,
    integrator,
    platform=platform,
)

simulation.context.setPositions(modeller.positions)

describe_state(
    simulation.context.getState(getEnergy=True, getForces=True),
    "Initial State",
)

print("Minimizing...")
simulation.minimizeEnergy(
    tolerance=0.01 * kilojoule / (nanometer * mole),
    maxIterations=1000,
)

describe_state(
    simulation.context.getState(getEnergy=True, getForces=True),
    "Minimized State",
)


# =========================
# Metadynamics
# =========================

# Define torsion collective variables
cv1 = CustomTorsionForce("theta")
cv1.addTorsion(16, 17, 18, 26)
phi = BiasVariable(cv1, -np.pi, np.pi, 0.5, True)

cv2 = CustomTorsionForce("theta")
cv2.addTorsion(5, 16, 17, 18)
psi = BiasVariable(cv2, -np.pi, np.pi, 0.5, True)

meta = Metadynamics(
    system,
    [phi, psi],
    500 * kelvin,
    10.0,
    1.0 * kilojoules_per_mole,
    100,
)

# Reinitialize simulation for meta
integrator = openmm.LangevinIntegrator(
    500 * kelvin,
    1.0 / picoseconds,
    0.001 * picoseconds
)

simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

simulation.reporters.append(
    PDBReporter("meta_dynamics.pdb", 1000)
)
simulation.reporters.append(
    StateDataReporter(
        "meta_dynamics.csv",
        100,
        step=True,
        potentialEnergy=True,
        temperature=True,
    )
)

print("Running metadynamics...")
meta.step(simulation, 500000)

# Plot free energy surface
plt.imshow(meta.getFreeEnergy())
plt.show()
